# Glider data [^download]

[^download]: This notebook can be downloaded as **{nb-download}
repr(example_glider.ipynb) **.

This notebook demonstrates how to combine glider navigation with acoustic detections: load the navigation track, plot selected or all detection labels against depth, and compute acoustic diversity. The bundled example data are from an autonomous glider survey.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from osekit.utils.timestamp import strptime_from_text
from pandas import Timedelta, Timestamp, read_csv

from disclose.utils.filtering import read_dataframe
from disclose.utils.glider import (
    compute_acoustic_diversity,
    load_glider_nav,
    plot_detections_with_nav_data,
)

## Load data

Read the bundled navigation files and detection results, then restrict the navigation track to the period covered by this example.

In [ ]:
input_dir = Path(r"_static/OHAGEODAMS_nav")
nav = load_glider_nav(input_dir)
nav = nav[nav["Timestamp"] > Timestamp("2024-02-21", tz="UTC")]
df = read_dataframe(Path(r"_static/OHAGEODAMS_results.csv"))

## Plot detections with navigation data

First select two labels to focus the plot; the next cell passes the full detection table to show every label. Both plots use depth as the navigation criterion.

In [ ]:
# plot specific labels
plot_detections_with_nav_data(
    df=df[df["label"].isin(["Sperm_Whale", "Delphinids clics"])],
    nav=nav,
    criterion="Depth",
    ticks=Timedelta("1D"),
)
plt.show()

In [ ]:
# plot all labels
plot_detections_with_nav_data(
    df=df,
    nav=nav,
    criterion="Depth",
    ticks=Timedelta("1D"),
)
plt.show()

## Compute acoustic diversity

Build the detection time vector from the result filenames, localize it to UTC, and use it with the navigation track to calculate acoustic diversity.

In [ ]:
task_status_filenames = read_csv(Path(r"_static/OHAGEODAMS_results.csv"))["filename"]
time_vector = [
    strptime_from_text(f, "%Y_%m_%d_%H_%M_%S").tz_localize("UTC")
    for f in task_status_filenames
]

df_acoustic_div = compute_acoustic_diversity(df=df, nav=nav, time_vector=time_vector)